# Brain Tumor Classification - Model Training Notebook

Berdasarkan hasil Exploratory Data Analysis (EDA) sebelumnya, kita akan melakukan preprocessing data tingkat lanjut (CLAHE & Padding) sebelum memasukkannya ke dalam Model *Convolutional Neural Network* (CNN). 

Notebook ini dirancang khusus untuk dijalankan di **Google Colab** dengan pemanfaatan GPU.

In [ ]:
import os
import cv2
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
from glob import glob

# Keras / TensorFlow
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, Input, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

# Memastikan TensorFlow mendeteksi GPU di Google Colab
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


## 1. Physical Data Preprocessing: CLAHE & Zero Padding
Fungsi ini membaca gambar mentah (`brain_tumor_dataset`), melakukan pemerataan kontras cahaya (CLAHE), menambahkan padding agar gambar menjadi persegi panjang sempurna 224x224 (tanpa distorsi), lalu menyimpannya ke folder baru bernama `dataset_processed`. Proses ini cukup dijalankan **satu kali saja**.

In [ ]:
# Path Dataset (Sesuaikan dengan lokasi di Google Drive / Local jika berbeda)
RAW_DATA_PATH = 'brain_tumor_dataset'
PROCESSED_DATA_PATH = 'dataset_processed'
CLASSES = ['no', 'yes']
TARGET_SIZE = 224

pdef resize_with_pad(img_cv2, expected_size):
    ''' Mengubah ukuran rasio OpenCV ke persegi dengan pad hitam '''
    old_size = img_cv2.shape[:2]
    ratio = float(expected_size) / max(old_size)
    new_size = tuple([int(x * ratio) for x in old_size])
    
    # Resize sesuai rasio
    img_resized = cv2.resize(img_cv2, (new_size[1], new_size[0]), interpolation=cv2.INTER_LANCZOS4)
    
    # Buat kanvas hitam (Zero Padding)
    delta_w = expected_size - new_size[1]
    delta_h = expected_size - new_size[0]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    
    color = [0, 0, 0]
    img_padded = cv2.copyMakeBorder(img_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return img_padded

# Eksekusi Transformasi Massal
if not os.path.exists(PROCESSED_DATA_PATH):
    os.makedirs(PROCESSED_DATA_PATH)
    
    for cl in CLASSES:
        os.makedirs(os.path.join(PROCESSED_DATA_PATH, cl), exist_ok=True)
        img_paths = glob(os.path.join(RAW_DATA_PATH, cl, '*'))
        
        print(f"Memproses folder '{cl}'... ({len(img_paths)} gambar)")
        for path in tqdm(img_paths):
            filename = os.path.basename(path)
            # Membaca sebagai Grayscale dulu karena MRI umumnya hitam putih
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue
                
            # 1. Terapkan CLAHE untuk perbaikan kontras
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            img_clahe = clahe.apply(img)
            
            # Convert kembali ke BGR (3 Channel) yang diminta oleh kebanyakan model Transfer Learning CNN
            img_clahe_bgr = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2BGR)
            
            # 2. Resizing & Padding (Mencegah distorsi memanjang/melebar)
            img_final = resize_with_pad(img_clahe_bgr, TARGET_SIZE)
            
            # Simpan Data Fisik
            save_path = os.path.join(PROCESSED_DATA_PATH, cl, filename)
            # Ekstensi diubah jadi JPG untuk kepastian
            if not save_path.endswith('.jpg'):
                save_path = save_path.split('.')[0] + '.jpg'
                
            cv2.imwrite(save_path, img_final)
    print("\nFolder 'dataset_processed' selesai dibuat! Dataset baru sudah matang.")
else:
    print("\nFolder 'dataset_processed' sudah ada. Melewati proses...")

## 2. Keras ImageDataGenerator (Augmentasi Data Ideal)
Sekarang kita konfigurasikan `ImageDataGenerator` untuk mengambil data lansung dari `dataset_processed`. Sesuai _insight_ EDA, kita hanya menerapkan sedikit simulasi augmentasi natural seperti memiringkan (*tilt*) otak sedikit tanpa menarik melar (*shear*).

In [ ]:
BATCH_SIZE = 32

# Generator Train (Data Augmentation Diformulasikan secara ideal untuk gambar medis MRI)
train_datagen = ImageDataGenerator(
    rescale=1./255,               # Standar normalisasi agar rentang pixel 0-1
    rotation_range=10,            # Rotasi 10 derajat lazim dengan kelurusan posisi pasien di mesin MRI
    width_shift_range=0.1,        # Geser maksimum 10%
    height_shift_range=0.1,       
    zoom_range=0.1,               # Zoom tak boleh berlebih agar tumor tak tercrop ke luar bingkai
    horizontal_flip=True,         # Flip kanan kiri otak diperbolehkan (simetris)
    validation_split=0.2          # 80% Training, 20% Validation
)

# Generator untuk Validation (TIDAK BOLEH DIAUGMENTASI!)
val_datagen = ImageDataGenerator(
    rescale=1./255, 
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    PROCESSED_DATA_PATH,
    target_size=(TARGET_SIZE, TARGET_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',          # Binary karena Yes vs No
    subset='training',
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    PROCESSED_DATA_PATH,
    target_size=(TARGET_SIZE, TARGET_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=42,
    shuffle=False                 # WAJIB FALSE agar evaluasi metrik akhir berurutan akurat
)

## 3. Pembangunan Arsitektur Model CNN
Karena dataset ini jumlah sampel aslinya relatif terbatas (~250 gambar), kita akan menggunakan Custom CNN dipadukan lapisan `BatchNormalization` dan `Dropout` ekstrem menghindari *Overfitting* akibat data yang sedikit.

In [ ]:
model = Sequential([
    Input(shape=(TARGET_SIZE, TARGET_SIZE, 3)),
    
    # Convolutional Block 1
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    
    # Convolutional Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    
    # Convolutional Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    
    # Flatten to Fully Connected Network
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Sigmoid untuk klasifikasi Biner (Yes/No Tumor)
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. Melatih Model (Training Phase)
Kita terapkan _EarlyStopping_ otomatis. Prosesnya akan berhenti begitu parameter loss pada validation mulai naik (*overfitting*).

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

EPOCHS = 35

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

## 5. Visualisasi Hasil Pelatihan (Loss & Accuracy Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy Plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
axes[0].set_title('Model Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Loss Plot
axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation Loss', marker='o')
axes[1].set_title('Model Loss')
axes[1].set_ylabel('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluasi Metrik & Confusion Matrix Akhir
Mari kita evaluasi performa model ini pada Test Set (Validation Set yang belum pernah dilihat model di tahapan *training*).

In [ ]:
# Reset generator agar selaras prediksi urut
val_generator.reset()

# Melakukan prediksi probabilitas ke data validasi
predictions = model.predict(val_generator)

# Mengonversi probabilitas sigmoid menjadi binar (0 atau 1) dengan threshold 0.5
y_pred = (predictions > 0.5).astype(int)
y_true = val_generator.classes

print("================ CLASSIFICATION REPORT ================")
print(classification_report(y_true, y_pred, target_names=['Normal (No)', 'Tumor (Yes)']))

In [ ]:
# Menampilkan Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal (No)', 'Tumor (Yes)'],
            yticklabels=['Normal (No)', 'Tumor (Yes)'])
plt.title('Confusion Matrix - Prediksi Validasi')
plt.ylabel('Label Sebenarnya (True)')
plt.xlabel('Prediksi Model (Predicted)')
plt.show()